pip install pyarrow --break-system-packages

In [1]:
"""
Moteur d'indexation d'images — version Spark
=============================================
Migration de l'indexation séquentielle (pandas + boucle for) vers Spark.
 
Principe :
  - Chaque image = 1 ligne dans un DataFrame Spark
  - cv2 s'exécute dans les workers via mapInPandas (contourne la JVM)
  - Les histogrammes sont calculés en parallèle sur tous les cœurs
  - Le résultat est sauvegardé en CSV (compatible avec l'original) ET en Parquet
 
Usage :
    python3 image_indexer_spark.py --path /chemin/vers/images --methode HistGREY --bins 16
    python3 image_indexer_spark.py --path /chemin/vers/images --methode HistRGB  --bins 4
"""
 
import os
import sys
import argparse
import pandas as pd
 
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType
)

In [2]:
# ─────────────────────────────────────────────────────────────
# SESSION SPARK
# ─────────────────────────────────────────────────────────────
 
spark = (
    SparkSession.builder
    .appName("ImageIndexer")
    .master("local[*]")
    # Arrow accélère les échanges Python ↔ JVM pour mapInPandas
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [3]:
# ─────────────────────────────────────────────────────────────
# ÉTAPE 1 — Lister les images (remplace CreateFiles_base10000)
# ─────────────────────────────────────────────────────────────
 
def create_image_dataframe(path: str):
    """
    Remplace CreateFiles_base10000.
    Crée un DataFrame Spark — 1 ligne par image, 1 colonne filepath.
 
    PySpark 4.x : schéma explicite obligatoire.
    L'inférence automatique sur des tuples a été supprimée
    (erreur legacyInferArrayTypeFromFirstElement).
    """
    files = [
        os.path.join(path, f)
        for f in os.listdir(path)
        if not f.startswith(".")
        and f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff"))
    ]
 
    if not files:
        raise ValueError(f"Aucune image trouvée dans : {path}")
 
    print(f"  {len(files)} images trouvées")
 
    schema = StructType([StructField("filepath", StringType(), False)])
    df = spark.createDataFrame([Row(filepath=f) for f in files], schema=schema)
    return df

# ─────────────────────────────────────────────────────────────
# ÉTAPE 2 — UDFs de calcul d'histogramme
# ─────────────────────────────────────────────────────────────
# Pourquoi des UDF ?
# cv2 est une lib Python/C++ — Spark ne sait pas l'appeler directement.
# Une UDF (User Defined Function) permet d'exécuter du code Python
# sur chaque ligne d'un DataFrame Spark, en parallèle sur tous les workers.
# ─────────────────────────────────────────────────────────────

def _compute_gray_hist(filepath: str, bins: int) -> list:
    """
    Version interne de ComputeGrayLevelHistogram.
    Retourne une liste Python (sérialisable par Spark).
    """
    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if image is None:
        # Image illisible : on retourne un vecteur nul plutôt que planter le job
        return [0.0] * bins
    hist = cv2.calcHist([image], [0], None, [bins], [0, 256])
    hist = cv2.normalize(hist, hist)
    return hist.flatten().tolist()

def _compute_rgb_hist(filepath: str, bins: int) -> list:
    """
    Version interne de ComputeRGBHistogram.
    bins^3 valeurs (ex : 4^3 = 64, 6^3 = 216).
    """
    image = cv2.imread(filepath, cv2.IMREAD_COLOR)
    if image is None:
        return [0.0] * (bins ** 3)
    b = [bins, bins, bins]
    hist = cv2.calcHist([image], [0, 1, 2], None, b, [0, 256, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist)
    return hist.flatten().tolist()


def make_gray_udf(bins: int):
    """
    Fabrique une UDF Spark pour l'histogramme gris avec le nombre de bins fixé.
    On utilise une closure pour capturer bins (Spark ne passe pas de paramètres aux UDF).
    """
    @udf(returnType=ArrayType(FloatType()))
    def gray_udf(filepath: str):
        return _compute_gray_hist(filepath, bins)
    return gray_udf


def make_rgb_udf(bins: int):
    """
    Fabrique une UDF Spark pour l'histogramme RGB avec le nombre de bins fixé.
    """
    @udf(returnType=ArrayType(FloatType()))
    def rgb_udf(filepath: str):
        return _compute_rgb_hist(filepath, bins)
    return rgb_udf

In [4]:
# ─────────────────────────────────────────────────────────────
# ÉTAPE 2 — Calcul d'histogramme via mapInPandas
# ─────────────────────────────────────────────────────────────
# Pourquoi mapInPandas plutôt que UDF classique ?
#
# Les UDFs avec ArrayType déclenchent une sérialisation JVM ↔ Python
# via Py4J instable en PySpark 4.x → erreur "JavaPackage not callable".
#
# mapInPandas contourne complètement la JVM :
#   - Spark découpe le DataFrame en batches pandas
#   - Chaque batch est envoyé à un worker Python
#   - cv2 tourne dans ce process Python sans passer par Py4J
#   - Le résultat pandas revient directement à Spark
# ─────────────────────────────────────────────────────────────
 
def _make_hist_mapper(methode: str, bins: int):
    """
    Retourne une fonction compatible mapInPandas.
    La closure capture methode et bins — Spark ne peut pas passer
    de paramètres aux fonctions map, on utilise donc une closure Python.
 
    Entrée  : itérateur de DataFrames pandas (colonne 'filepath')
    Sortie  : itérateur de DataFrames pandas (filepath + h_0 ... h_N)
    """
    def mapper(iterator):
        # Imports dans le worker — chaque process Python doit importer cv2
        import cv2
        import numpy as np
        import pandas as pd
 
        for batch in iterator:
            records = []
            for filepath in batch["filepath"]:
 
                if methode == "HistGREY":
                    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
                    if image is None:
                        hist = np.zeros(bins, dtype=np.float32)
                    else:
                        hist = cv2.calcHist([image], [0], None, [bins], [0, 256])
                        hist = cv2.normalize(hist, hist).flatten()
 
                elif methode == "HistRGB":
                    image = cv2.imread(filepath, cv2.IMREAD_COLOR)
                    n = bins ** 3
                    if image is None:
                        hist = np.zeros(n, dtype=np.float32)
                    else:
                        b = [bins, bins, bins]
                        hist = cv2.calcHist(
                            [image], [0, 1, 2], None, b,
                            [0, 256, 0, 256, 0, 256]
                        )
                        hist = cv2.normalize(hist, hist).flatten()
 
                # 1 ligne = chemin + valeurs de l'histogramme à plat
                row = {"filepath": filepath}
                for i, v in enumerate(hist):
                    row[f"h_{i}"] = float(v)
                records.append(row)
 
            yield pd.DataFrame(records)
 
    return mapper

# ─────────────────────────────────────────────────────────────
# ÉTAPE 3 — Indexation (remplace IndexDataBase)
# ─────────────────────────────────────────────────────────────

def index_database(path: str, methode: str, bins: int):
    """
    Remplace IndexDataBase.

    Différences clés vs l'original :
      - Plus de boucle for  → Spark distribue sur tous les cœurs
      - Plus de pd.concat() → coûteux et séquentiel
      - Sauvegarde en CSV (compatible) ET en Parquet (requêtable)

    Sorties :
      Base10000.HistGREY_16.csv     — compatible avec l'index original
      Base10000.HistGREY_16.parquet — format requêtable efficacement
    """

    print(f"\n  Méthode : {methode} | Bins : {bins}")

    # 1. Lister les images → DataFrame Spark
    df_images = create_image_dataframe(path)

    # 2. Choisir la bonne UDF selon la méthode
    if methode == "HistGREY":
        compute_hist = make_gray_udf(bins)
        expected_size = bins
    elif methode == "HistRGB":
        compute_hist = make_rgb_udf(bins)
        expected_size = bins ** 3
    else:
        sys.exit("methode doit être 'HistGREY' ou 'HistRGB'")

    print(f"  Taille du vecteur histogramme : {expected_size} valeurs")

    # 3. Appliquer l'UDF — Spark distribue le calcul sur tous les cœurs
    #    Chaque worker calcule l'histogramme de son lot d'images
    print("  Calcul des histogrammes en parallèle...")
    df_indexed = df_images.withColumn(
        "histogram",
        compute_hist(col("filepath"))
    )

    # 4. Exploser le vecteur en colonnes nommées h_0, h_1, ..., h_N
    #    (format compatible avec l'index original : 1 valeur par colonne)
    col_names = [f"h_{i}" for i in range(expected_size)]
    df_result = df_indexed.select(
        col("filepath"),
        *[col("histogram")[i].alias(col_names[i]) for i in range(expected_size)]
    )

    # 5. Sauvegarder
    base_name = f"Base10000.{methode}_{bins}"
    csv_path     = os.path.join(os.getcwd(), base_name + ".csv")
    parquet_path = os.path.join(os.getcwd(), base_name + ".parquet")

    print(f"  Sauvegarde CSV     → {csv_path}")
    (df_result
        .write
        .mode("overwrite")
        .option("header", "false")   # compatible avec l'original (pas d'en-tête)
        .csv(csv_path))

    print(f"  Sauvegarde Parquet → {parquet_path}")
    (df_result
        .write
        .mode("overwrite")
        .parquet(parquet_path))

    print(f"\n  Terminé — {df_result.count()} images indexées")
    return df_result

In [5]:
# ─────────────────────────────────────────────────────────────
# ÉTAPE 3 — Indexation (remplace IndexDataBase)
# ─────────────────────────────────────────────────────────────
 
def index_database(path: str, methode: str, bins: int):
    """
    Remplace IndexDataBase.
 
    Différences clés vs l'original :
      - Plus de boucle for  → mapInPandas distribue sur tous les cœurs
      - Plus de pd.concat() → coûteux et séquentiel
      - Sauvegarde en CSV (compatible) ET en Parquet (requêtable)
 
    Sorties :
      Base10000.HistGREY_16.csv     — compatible avec l'index original
      Base10000.HistGREY_16.parquet — format requêtable efficacement
    """
 
    print(f"\n  Méthode : {methode} | Bins : {bins}")
 
    # 1. Lister les images → DataFrame Spark
    df_images = create_image_dataframe(path)
 
    # 2. Taille du vecteur selon la méthode
    if methode == "HistGREY":
        expected_size = bins
    elif methode == "HistRGB":
        expected_size = bins ** 3
    else:
        sys.exit("methode doit être 'HistGREY' ou 'HistRGB'")
 
    print(f"  Taille du vecteur histogramme : {expected_size} valeurs")
 
    # 3. Schéma de sortie explicite — mapInPandas l'exige en PySpark 4.x
    out_schema = StructType(
        [StructField("filepath", StringType(), False)]
        + [StructField(f"h_{i}", FloatType(), True) for i in range(expected_size)]
    )
 
    # 4. Calcul distribué — cv2 tourne dans les workers Python
    print("  Calcul des histogrammes en parallèle...")
    mapper = _make_hist_mapper(methode, bins)
    df_result = df_images.mapInPandas(mapper, schema=out_schema)
 
    # 5. Sauvegarder
    base_name = f"Base10000.{methode}_{bins}"
    csv_path     = os.path.join(os.getcwd(), base_name + ".csv")
    parquet_path = os.path.join(os.getcwd(), base_name + ".parquet")
 
    print(f"  Sauvegarde CSV     → {csv_path}")
    (df_result
        .write
        .mode("overwrite")
        .option("header", "false")   # compatible avec l'original (pas d'en-tête)
        .csv(csv_path))
 
    print(f"  Sauvegarde Parquet → {parquet_path}")
    (df_result
        .write
        .mode("overwrite")
        .parquet(parquet_path))
 
    n = df_result.count()
    print(f"\n  Terminé — {n} images indexées")
    return df_result

In [6]:
# ─────────────────────────────────────────────────────────────
# POINT D'ENTRÉE
# ─────────────────────────────────────────────────────────────
PATH="D:/GitHub/imageSearchEngines/images"
METHODE="HistRGB"
BINS=256


df = index_database(
    path=PATH,
    methode=METHODE,
    bins=BINS,
)

print("\n  Aperçu (5 premières lignes) :")
df.show(5, truncate=True)

spark.stop()


  Méthode : HistRGB | Bins : 256
  10000 images trouvées


TypeError: 'JavaPackage' object is not callable